# Meta-analysis 
## Introduction


## Models of meta-analysis


## Meta-analysis of counted data
### Comparison of two independent proportions


#### Loading the data


In [ ]:
import io
import pandas as pd

raw_data_eyding = """
study,count_t,nobs_t,count_c,nobs_c
014,70,126,43,128
015,65,110,58,111
046,144,252,136,247
047,120,238,108,239
050,60,144,63,143
045,38,88,39,86
049,42,106,35,104
"""

data_eyding = pd.read_csv(
    io.StringIO(raw_data_eyding),
    converters={'study': lambda x: str(x)},
    index_col='study')

print("Raw data counts from the reboxetine study (Eyding):")
print(data_eyding)

#### Calculating effect sizes and variances


In [ ]:
import numpy as np
from statsmodels.stats.meta_analysis import effectsize_2proportions

# Extract data for effect size calculation
count_t = data_eyding['count_t'].values
nobs_t  = data_eyding['nobs_t'].values
count_c = data_eyding['count_c'].values
nobs_c  = data_eyding['nobs_c'].values

# Calculate effect sizes (log odds ratios) and variances
eff_eyding, var_eyding = effectsize_2proportions(
    count1=count_t,
    nobs1=nobs_t,
    count2=count_c,
    nobs2=nobs_c,
    statistic="odds-ratio",  # Could also use risk difference p1 - p2
)

print("Trials No.:\t", data_eyding.index.values)
print("Log odds ratios:", eff_eyding.round(2))
print("Log variance:\t", var_eyding.round(3))
print("Odds ratios:\t", np.exp(eff_eyding).round(3))
print("Variance:\t", np.exp(var_eyding).round(3))

In [ ]:
study_014 = data_eyding.loc[data_eyding.index == '014']
a, n1, c, n2 = study_014.iloc[0]

b, d = n1 - a, n2 - c
p1, p2 = a / n1, c / n2

log_or_014 = np.log(p1 / (1 - p1) / (p2 / (1 - p2)))
log_var_014 = (1 / a + 1 / b + 1 / c + 1 / d)

print("Effect size and variance for study No. 014:")
print(f" Log(OR) = {log_or_014:.2f} \
and Log(s²) = {log_var_014:.3f}")
print(f" OR = {np.exp(log_or_014):.3f} \
and s² = {np.exp(log_var_014):.3f}")

#### Calculating combined effects


In [ ]:
from statsmodels.stats.meta_analysis import combine_effects

# Perform meta-analysis (random-effects using chi2/DL method)
res_eyding = combine_effects(
    effect=eff_eyding,
    variance=var_eyding,
    method_re="chi2",  # Estimate for RE variance τ²
    use_t=False,
    row_names=data_eyding.index)

# Print formatted summary results
print(
    "Meta-analysis summary (log) of the reboxetine study (Eyding):",
    f"Random effects (RE) method: {res_eyding.method_re}",
    res_eyding.summary_frame(),
    sep='\n')

In [ ]:
# Extract relevant summary statistics and exponentiate for OR
res_eyding_or = np.exp(res_eyding.summary_frame().loc[
    :'random effect',  # Remove WLS by slicing
    ['eff', 'sd_eff', 'ci_low', 'ci_upp']
]).round(2)

print(
    "Meta-analysis summary (ORs) of the reboxetine study (Eyding):\n",
    res_eyding_or)

In [ ]:
from scipy.stats import norm

# Defining critical z value for α=0.05 (z_crit ≈ 1.96)
z_crit = norm.ppf(q=(1 - 0.05/2))

ci_log_or = np.array((
    log_or_014 - z_crit * np.sqrt(log_var_014),
    log_or_014 + z_crit * np.sqrt(log_var_014)))

print("Confidence interval for study No. 014:")
print(" 95% CI logOR:\t", ci_log_or.round(4))
print(" 95% CI OR:\t", np.exp(ci_log_or).round(2))

### Forest plot


In [ ]:

import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

fig = res_eyding.plot_forest(
    use_exp=True, # to show OR instead of logOR
)
fig.set_figwidth(3.5)
fig.set_figheight(3)

# Show a vertical line at OR=1
plt.axvline(x=1, ls='--', color='blue')
plt.xscale('log')

# By default, a logarithmic scale uses a LogFormatter
plt.gca().xaxis.set_major_formatter(ScalarFormatter())
plt.xticks([.33, .5, 1, 2, 3, 5])

plt.title("Forest plot -- reboxetine study (Eyding)")
plt.xlabel('Odds ratio (95% CI)');

### Fixed-effects (inverse-variance)
#### Weights and variances


In [ ]:
# Fixed-effect weights (inverse variance)
fe_weights_eyding = res_eyding.weights_fe
print(
    "\nFixed-effect weights (inverse variance):\n",
    fe_weights_eyding.round(3))

# Fixed-effect weights (normalized)
normalized_fe_weights_eyding = fe_weights_eyding / fe_weights_eyding.sum()
print(
    "Normalized fixed-effect weights:\n",
    normalized_fe_weights_eyding.round(3))

#### Pooled effect
#### Confidence interval (fixed-effect)


In [ ]:
# Calculate combined effect size (fixed-effect)
E_eyding_fe = (eff_eyding*fe_weights_eyding).sum()/fe_weights_eyding.sum()

# We can extract the pooled and individual effect sizes:
#E_eyding_fe = res_eyding.mean_effect_fe
#eff_eyding = res_eyding.effect

# Calculate variance and standard error (fixed-effect)
var_eyding_fe = 1 / np.sum(fe_weights_eyding)
SE_eyding_fe = var_eyding_fe**0.5

# Calculate confidence interval (fixed-effect)
ci_E_eyding_fe = np.array((
    E_eyding_fe - z_crit * SE_eyding_fe,
    E_eyding_fe + z_crit * SE_eyding_fe))

# If t-distribution is preferred, with k the number of studies
#from scipy.stats import t as t_dist
#t_crit = t_dist(df=res_eyding.df).ppf((1+.95)/2)

print(f"Combined (log) effect size (FE) = {E_eyding_fe:.3f} \
and SE = {SE_eyding_fe:.4f}")
print(" 95% CI of combined effect size (FE):", ci_E_eyding_fe.round(3))
print(f"Overall OR (FE) = {np.exp(E_eyding_fe).round(2)}")
print(" 95% CI of overall OR (FE):", np.exp(ci_E_eyding_fe).round(3))

### Mantel-Haenszel odds-ratio


In [ ]:
# Extract count data into arrays
t = data_eyding['count_t'].values  # treatment
nt = data_eyding['nobs_t'].values  # total treatment
c = data_eyding['count_c'].values  # control
nc = data_eyding['nobs_c'].values  # total control

# Construct the 2 x 2 x k contingency table, containing
# event and non-event counts, in treatment and control groups
counts = np.column_stack([t, nt - t, c, nc - c])
ctables = counts.T.reshape(2, 2, -1)

# Print the first study contingency table for verification
print(f"Contingency table for study No. {data_eyding.index[0]}:")
print(ctables[:, :, 0])

In [ ]:
import statsmodels.stats.api as smstats

# Create a StratifiedTable object from the contingency table; 
# the contingency table is cast to float64 to ensure compatibility
st = smstats.StratifiedTable(ctables.astype(np.float64))

# Calculate the pooled log OR
logodds_pooled = st.logodds_pooled

# Calculate the standard error of the pooled log OR
logodds_pooled_se = st.logodds_pooled_se

# Calculate the CI of the pooled log OR
logodds_pooled_ci = np.array(st.logodds_pooled_confint())

# Print the results with formatted output
print(f"Pooled LogOR (MH) = {logodds_pooled:.3f} \
and SE = {logodds_pooled_se:.4f}")
print(" 95% CI:", logodds_pooled_ci.round(4))

print(f"Pooled OR (MH) = {np.exp(logodds_pooled):.2f}")
print(" 95% CI of pooled OR (MH):", np.exp(logodds_pooled_ci).round(3))

In [ ]:
# Test the null hypothesis that the pooled odds ratio = 1 (log odds = 0)
print("MH test pooled OR=1:")
print(st.test_null_odds())

In [ ]:
print(st.summary())

### Heterogeneity


In [ ]:
from scipy.stats import chi2

Q_eyding = np.sum(fe_weights_eyding * (eff_eyding - E_eyding_fe)**2)
df_eyding = len(eff_eyding) - 1  # Same as res_eyding.df
p_heterogeneity_eyding = 1 - chi2.cdf(x=Q_eyding, df=df_eyding)

print("Homogeneity test for the reboxetine study (manual):")
print(f"Cochrane Q = {Q_eyding:.3f}, \
with P value for heterogeneity = {p_heterogeneity_eyding:.6f}")

In [ ]:
print("Result of the homogeneity test (statsmodels):")
print(res_eyding.test_homogeneity())

In [ ]:
print("Homogeneity test for the reboxetine study:")
print(f" I² (manual) = {(Q_eyding - df_eyding) / Q_eyding * 100:.2f}%")
print(f" I² (attribute) = {res_eyding.i2 * 100:.2f}%")

### Random-effects models 
#### Tau-squared (τ²)


In [ ]:
# Calculate τ² using fixed-effect weights
q_eyding = (
    np.sum(fe_weights_eyding * eff_eyding**2)
    - (np.sum(fe_weights_eyding * eff_eyding))**2
    / fe_weights_eyding.sum()
)
c_eyding = (
    np.sum(fe_weights_eyding)
    - np.sum(fe_weights_eyding**2)
    / np.sum(fe_weights_eyding)
)

tau2_eyding = (q_eyding - df_eyding) / c_eyding  # Same as res_eyding.tau2

print("τ² in the reboxetine study (Eyding) = ", round(tau2_eyding, 4))

# Calculate random-effect weights using τ²
re_weights_eyding = 1 / (var_eyding + tau2_eyding)
#re_weights_eyding = res_eyding.weights_re

# Calculate normalized random-effect weights
normalized_re_weights_eyding = re_weights_eyding / re_weights_eyding.sum()

print("Random-effect weights:\n", re_weights_eyding.round(3))
print(
    "Normalized random-effect weights:\n",
    normalized_re_weights_eyding.round(3))

#### Confidence intervals (random-effects)


In [ ]:
# Calculate combined effect size (random-effect)
E_eyding_re = (
    (eff_eyding * re_weights_eyding).sum()
    /
    re_weights_eyding.sum()
)
#E_eyding_re = (eff_eyding * normalized_re_weights_eyding).sum()

# Note that we can extract the effect size and weights as follows:
# E_eyding_re = res_eyding.mean_effect_re
# re_weights_eyding = res_eyding.weights_re

# Calculate variance and standard error (random-effect)
var_eyding_re = 1 / sum(re_weights_eyding)
SE_eyding_re = var_eyding_re**0.5

# Calculate confidence interval (random-effect)
ci_E_eyding_re = np.array((
    E_eyding_re - z_crit * SE_eyding_re,
    E_eyding_re + z_crit * SE_eyding_re))

print(f"Combined (log) effect size (RE) = {E_eyding_re:.3f} \
and SE = {SE_eyding_re:.4f}")
print(" 95% CI of combined effect size (RE):", ci_E_eyding_re.round(4))
print(f"Overall OR (RE) = {np.exp(E_eyding_re):.2f}")
print(" 95% CI of overall OR (RE):", np.exp(ci_E_eyding_re).round(2))

#### Confidence intervals using the t-distribution


In [ ]:
# Meta-analysis using the t-distribution
res_eyding_t = combine_effects(
    effect=eff_eyding,
    variance=var_eyding,
    method_re="chi2",
    use_t=True,
    row_names=data_eyding.index)

res_eyding_t.conf_int_samples(
    nobs=np.array(  # For calculating DF
        data_eyding['nobs_t'] + data_eyding['nobs_c']))

# Print formatted summary results
print(
    "Meta-analysis summary (ORs) using the t-distribution:",
    f"Random effects (RE) method: {res_eyding_t.method_re}",
    np.exp(
        res_eyding_t.summary_frame().iloc[:-2,]  # Removed WLS rows
    ).round(2),
    sep='\n')

In [ ]:
from scipy.stats import t as t_dist

t_crit_eyding = t_dist(df=res_eyding.df).ppf((1+.95)/2)

ci_t_E_eyding_re = np.array((
    E_eyding_re - t_crit_eyding * SE_eyding_re,
    E_eyding_re + t_crit_eyding * SE_eyding_re))

print(f"Combined effect size (RE) = {E_eyding_re:.3f} \
and SE = {SE_eyding_re:.4f}")
print(" 95% CI using the t-distribution:", ci_t_E_eyding_re.round(4))
print(f"Overall OR (RE) = {np.exp(E_eyding_re):.2f}")
print(
    " 95% CI of the total OR (RE) using the t-distribution:",
    np.exp(ci_t_E_eyding_re).round(2))

### Advanced random-effects models


## Meta-analysis of continuous data
### Overall effect size


### Continuous data


In [ ]:
# Define the row names (lab identifiers)
rownames_kacker= np.array(
    ['PTB', 'NMi', 'NIMC', 'KRISS', 'LGC', 'NRC', 'IRMM', 'NIST', 'LNE'])
# Define the array of mean values for each laboratory
mean_kacker = np.array(
    [61.00, 61.40, 62.21, 62.30, 62.34, 62.60, 62.70, 62.84, 65.90])
# Define the array of standard deviation values
std_kacker = np.array(
    [0.45, 1.10 , 0.30, 0.45 , 0.62, 0.75, 0.26, 0.15, 1.35])

### Pooled effect size 


In [ ]:
res_kacker = combine_effects(
    effect=mean_kacker,
    variance=std_kacker**2,  # variance
    method_re='dl',           # Same as 'chi2'
    use_t=False,
    row_names=rownames_kacker)

# Print formatted summary results
print(
    "Meta-analysis summary of the lead study (Kacker):",
    f"Random effects (RE) method: {res_kacker.method_re}",
    res_kacker.summary_frame().iloc[:-2,].round(2),
    sep='\n')

### Forest plot using continuous data


In [ ]:

fig = res_kacker.plot_forest(
    use_exp=False,  # Linear scale is only required
)
fig.set_figwidth(3.5)
fig.set_figheight(3)

plt.title('Forest plot -- lead study (Kacker)')
plt.xlabel('[Pb] (nmol/kg)');

### Fixed-effects


In [ ]:
# Calculate the weighted effects by multiplying 
# each effect by its fixed-effects weight
kacker_weighted = mean_kacker * res_kacker.weights_fe
#kacker_weighted = res_kacker.effect * res_kacker.weights_fe

# Calculate the pooled fixed effect by summing the weighted effects 
# and dividing by the sum of the weights
E_kacker_fe = kacker_weighted.sum() / res_kacker.weights_fe.sum()

# Calculate inverse variances and standard error (fixed-effect)
inverse_variance_kacker = 1 / std_kacker**2
SE_kacker_fe = 1 / np.sqrt(inverse_variance_kacker.sum())

# Calculate confidence interval (fixed-effect)
ci_E_kacker_fe = np.array((
    E_kacker_fe - z_crit * SE_kacker_fe,
    E_kacker_fe + z_crit * SE_kacker_fe))

# Print the results
print(f"Combined (mean) effect size (FE) = {E_kacker_fe:.5f} \
and SE = {SE_kacker_fe:.4f}")
print(
    " 95% CI of combined (mean) effect size (FE):",
    ci_E_kacker_fe.round(3))

### Heterogeneity using continuous data


In [ ]:
print("Cochrane heterogeneity assessment in the lead study (Kacker):")
print(res_kacker.test_homogeneity())
print()
print(f"I² = {res_kacker.i2 * 100:.2f}%")

### Random-effects


In [ ]:
# Print τ²
tau2_kacker = res_kacker.tau2
print("τ² in the lead study (Kacker) = ", round(tau2_kacker, 4))

print("Random-effect weights:\n", res_kacker.weights_re.round(3))

# Calculate normalized random-effect weights
normalized_re_weights_kacker = (
    res_kacker.weights_re / res_kacker.weights_re.sum())
print(
    "Normalized random-effect weights:\n",
    normalized_re_weights_kacker.round(3))

In [ ]:
# Calculate combined effect size (RE)
E_kacker_re = (mean_kacker * normalized_re_weights_kacker).sum()

# Calculate inverse variances and standard error (RE)
re_weights_kacker = 1 / (std_kacker**2 + tau2_kacker)
#re_weights_kacker = res_kacker.weights_re
SE_kacker_re = 1 / np.sqrt(re_weights_kacker.sum())

# Calculate CI (RE)
ci_E_kacker_re = np.array((
    E_kacker_re - z_crit * SE_kacker_re,
    E_kacker_re + z_crit * SE_kacker_re))

print(f"Overall (mean) effect size (RE) = {E_kacker_re:.3f} \
and SE = {SE_kacker_re:.4f}")
print(" 95% CI:", ci_E_kacker_re.round(4))

### Standardized mean difference


In [ ]:
# Define the row names (study identifiers)
study_smd = np.array(
    ['Carroll', 'Grant', 'Peck', 'Donat', 'Stewart', 'Young'])

# Define the array of mean values in the treatment group
mean_t_smd = np.array([94, 98, 98, 94, 98, 96])
# Define the array of standard deviation values in the treatment group
sd_t_smd = np.array([22, 21, 28, 19, 21, 21])
# Define the array of number of observations in the treatment group
n_t_smd = np.array([60, 65, 40, 200, 50, 85])

# Define the arrays for the control group
mean_c_smd = np.array([92, 92, 88, 82, 88, 92])
sd_c_smd = np.array([20, 22, 26, 17, 22, 22])
n_c_smd = np.array([60, 65, 40, 200, 45, 85])

In [ ]:
from statsmodels.stats.meta_analysis import effectsize_smd

# Computing bias corrected estimate of standardized mean difference
# and estimate of variance of smd_bc
smd_bc, var_smdbc = effectsize_smd(
    mean1=mean_t_smd,
    sd1=sd_t_smd,
    nobs1=n_t_smd,
    mean2=mean_c_smd,
    sd2=sd_c_smd,
    nobs2=n_c_smd)

In [ ]:
res_smb = combine_effects(
    effect=smd_bc,
    variance=var_smdbc,
    method_re='dl',
    row_names=study_smd)

# Print formatted summary results
print(
    "Meta-analysis summary SMD):",
    f"Random effects (RE) method: {res_smb.method_re}",
    res_smb.summary_frame().iloc[:-2,].round(4),
    sep='\n')

In [ ]:

fig = res_smb.plot_forest(
    use_exp=False,  # Linear scale is only required
)
fig.set_figwidth(3.5)
fig.set_figheight(3)

plt.title('Forest plot -- SMD example')
plt.xlabel("Hedges g (95% CI)");

## Conclusion
